In [1]:
import os
os.chdir("../")

In [2]:
%pwd

'd:\\project\\Sleep Effiency App'

In [4]:
import dagshub
dagshub.init(repo_owner='PhamAnhTienn', repo_name='Sleep-Efficiency-App', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

c:\Users\PC\anaconda3\envs\Sleep_Effiency_App\lib\site-packages\rich\live.py:229: UserWarning: install "ipywidgets"
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=1def8e47-2aa6-4fd3-bebd-6ae488e06c76&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=aaef7a328fffddc2954a9654346740a5f9500d796e4362ee04da22c40e87cf1a




Accessing as PhamAnhTienn

Initialized MLflow to track repo "PhamAnhTienn/Sleep-Efficiency-App"

Repository PhamAnhTienn/Sleep-Efficiency-App initialized!

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [6]:
from Sleep_Efficiency_App.constants import *
from Sleep_Efficiency_App.utils.common import read_yaml, create_directories, save_json

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):
    
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.XGBoost
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metric_file_name=config.metric_file_name,
            target_column=schema.name,
            mlflow_uri="https://dagshub.com/PhamAnhTienn/Sleep-Efficiency-App.mlflow"
        )

        return model_evaluation_config

In [9]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import joblib

In [10]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
        
    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    
    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        X_test = test_data.drop([self.config.target_column], axis=1)
        y_test = test_data[self.config.target_column]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            predicted_qualities = model.predict(X_test)

            (rmse, mae, r2) = self.eval_metrics(y_test, predicted_qualities)

            # Saving metrics as local
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)

            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information
                mlflow.sklearn.log_model(model, "model", registered_model_name="XGBRegressorModel")
            else:
                mlflow.sklearn.log_model(model, "model")

In [11]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e

[2024-07-28 23:14:00,914: INFO: common: yaml file: config\config.yaml loaded successfully]
[2024-07-28 23:14:00,917: INFO: common: yaml file: params.yaml loaded successfully]
[2024-07-28 23:14:00,921: INFO: common: yaml file: schema.yaml loaded successfully]
[2024-07-28 23:14:00,923: INFO: common: created directory at: artifacts]
[2024-07-28 23:14:00,925: INFO: common: created directory at: artifacts/model_evaluation]
[2024-07-28 23:14:02,060: INFO: common: json file saved at: artifacts\model_evaluation\metrics.json]


c:\Users\PC\anaconda3\envs\Sleep_Effiency_App\lib\site-packages\_distutils_hack\__init__.py:26: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
Successfully registered model 'XGBRegressorModel'.
2024/07/28 23:14:14 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: XGBRegressorModel, version 1
Created version '1' of model 'XGBRegressorModel'.
